# Notebook de Generación de Embeddings con Model Serving Endpoint

**Versión Final**

Este notebook realiza el siguiente proceso:
1. Conecta al Model Registry y descarga dependencias del modelo.
2. Instala dependencias en el entorno del notebook.
3. Carga la tabla de origen con datos de texto (`chunks`).
4. Llama a un Model Serving Endpoint para generar embeddings por lote.
5. Guarda los resultados en una tabla Delta final.
6. La tabla final está lista para crear un índice de Vector Search en Databricks.

### Celda 1: Configuración Principal

Define todas las variables necesarias. **Por favor, modifica estos valores.**

In [0]:
# Widgets de configuración
dbutils.widgets.text("catalog_name", "bluetab", "Catalog Name")
dbutils.widgets.text("schema_name", "rag", "Schema Name")
dbutils.widgets.text("source_table_name", "docs_text", "Source Table Name")
dbutils.widgets.text("destination_table_name", "docs_text_embeddings", "Destination Table Name")
dbutils.widgets.text("endpoint_url", "https://dbc-ad7d5e59-0280.cloud.databricks.com/serving-endpoints/simple_embbeding/invocations", "Endpoint URL")
dbutils.widgets.text("databricks_token", "", "Databricks Token")

# MLflow Configuration
dbutils.widgets.text("experiment_name", "/Shared/RAG_Databricks_Bluetab_Pipeline", "MLflow Experiment Name")

# MLflow run management
dbutils.widgets.text("parent_run_id", "", "Parent Run ID")
dbutils.widgets.text("current_run", "", "Current Run")

In [0]:
# Obtener valores de los widgets y definir variables principales
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
SOURCE_TABLE_NAME = dbutils.widgets.get("source_table_name")
DESTINATION_TABLE_NAME = dbutils.widgets.get("destination_table_name")
ENDPOINT_URL = dbutils.widgets.get("endpoint_url")
DATABRICKS_TOKEN = dbutils.widgets.get("databricks_token")

# MLflow configuration
EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")

# Variables globales para gestión de parent/child runs
PARENT_RUN_ID = dbutils.widgets.get("parent_run_id") or None
CURRENT_RUN = dbutils.widgets.get("current_run") or None

source_table_fullname = f"{CATALOG_NAME}.{SCHEMA_NAME}.{SOURCE_TABLE_NAME}"
destination_table_fullname = f"{CATALOG_NAME}.{SCHEMA_NAME}.{DESTINATION_TABLE_NAME}"

print("Configuración cargada.")
print(f"Tabla de origen: {source_table_fullname}")
print(f"Tabla de destino: {destination_table_fullname}")
print(f"URL de Endpoint: {ENDPOINT_URL}")

In [0]:
# ---- Construcción de nombres completos (No necesitas modificar esto) ----
source_table_fullname = f"{CATALOG_NAME}.{SCHEMA_NAME}.{SOURCE_TABLE_NAME}"
destination_table_fullname = f"{CATALOG_NAME}.{SCHEMA_NAME}.{DESTINATION_TABLE_NAME}"

print("Configuración cargada.")
print(f"Tabla de origen: {source_table_fullname}")
print(f"Tabla de destino: {destination_table_fullname}")
print(f"URL de Endpoint: {ENDPOINT_URL}")

In [0]:
%run "./00 Configuration and Utils"

In [0]:
start_child_run("07_create_embeddings")

### Celda 3: Cargar Librerías y Configurar Conexión al Endpoint

In [0]:
import requests
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, FloatType

# Cargamos el token de acceso desde Databricks Secrets
print("Librerías importadas y token de acceso cargado.")

### Celda 4: Cargar la Tabla de Origen

In [0]:
try:
  source_df = spark.read.table(source_table_fullname)
  print(f"Tabla '{source_table_fullname}' cargada exitosamente.")
  print(f"Número de filas a procesar: {source_df.count()}")
  display(source_df.limit(5))
except Exception as e:
  print(f"Error al cargar la tabla '{source_table_fullname}'. Verifica que el nombre y los permisos son correctos.")
  raise e

### Celda 5: Definir Pandas UDF para Llamar al Endpoint en Lotes

Esta es la función principal que se aplicará a los datos de Spark. Opera en lotes para mayor eficiencia.

In [0]:
import json

def get_embeddings_from_endpoint(text_series: pd.Series) -> pd.Series:
    """
    Toma una serie de pandas con textos, llama al endpoint en un solo POST y devuelve una serie con los embeddings.
    """
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json",
    }
    data = {"inputs": text_series.tolist()}
    try:
        response = requests.post(ENDPOINT_URL, headers=headers, json=data, timeout=60)
        response.raise_for_status()
        predictions = response.json().get("predictions", [])
        return pd.Series(predictions)
    except requests.exceptions.RequestException as e:
        print(f"Error en la llamada a la API: {e}")
        return pd.Series([None] * len(text_series))
    
# Registrar la función como una Pandas UDF
embeddings_from_endpoint_udf = F.pandas_udf(get_embeddings_from_endpoint, returnType=ArrayType(FloatType()))

print("Pandas UDF para llamar al endpoint creada exitosamente.")

### Celda 6: Aplicar la UDF y Generar los Embeddings

In [0]:
print("Aplicando la UDF al DataFrame para generar embeddings desde el endpoint...")
print("Este proceso puede tardar.")

df_with_embeddings = source_df.withColumn(
    "embedding",
    embeddings_from_endpoint_udf(F.col("text"))
)

successful_embeddings_df = df_with_embeddings.filter(F.col("embedding").isNotNull())
failed_rows_count = df_with_embeddings.filter(F.col("embedding").isNull()).count()

print(f"Embeddings generados. Número de filas exitosas: {successful_embeddings_df.count()}")
if failed_rows_count > 0:
    print(f"ADVERTENCIA: Hubo {failed_rows_count} filas que no pudieron ser procesadas.")

display(successful_embeddings_df.limit(10))

### Celda 7: Guardar la Tabla Resultante

Guarda el DataFrame con los embeddings en una nueva tabla Delta, lista para Vector Search.

In [0]:
successful_embeddings_df.printSchema()

from pyspark.sql.functions import expr

print(f"Guardando la tabla con embeddings en: '{destination_table_fullname}'...")
successful_embeddings_df = successful_embeddings_df.withColumn(
    "embedding",
    expr("transform(embedding, x -> cast(x as float))")
)
successful_embeddings_df.write.format("delta").mode("overwrite").saveAsTable(destination_table_fullname)

print("¡Proceso completado exitosamente!")

In [0]:
successful_embeddings_df.printSchema()
successful_embeddings_df.selectExpr("typeof(embedding[0]) as tipo_elemento").distinct().show()

In [0]:
%sql
ALTER TABLE `bluetab`.`rag`.`docs_text_embeddings` SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
from pyspark.sql.functions import size, col
dimension_row = successful_embeddings_df.select(size(col("embedding")).alias("embedding_dimension")).first()
if dimension_row:
  embedding_dimension = dimension_row["embedding_dimension"]
  print(f"La dimensión del embedding es: {embedding_dimension}")
else:
  print("No se pudo determinar la dimensión. ¿La tabla está vacía?")